In [15]:
from pathlib import Path

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

## SPARQL Queries on the Agro Geo Knowledge Graph

This notebook demonstrates how the generated knowledge graph can be queried using SPARQL.
The queries combine semantic information, spatial relationships, and derived attributes such as NDVI values.

The knowledge graph contains information derived from:
- agricultural vector data
- OpenStreetMap infrastructure data
- NDVI raster analysis

In [44]:
from rdflib import Graph, Namespace
import pandas as pd

In [45]:
# import data / file paths

BASE_DIR = Path.cwd().parent  

OUTPUT_DIR = BASE_DIR / "data" / "processed"

INPUT_FILE_RDF_AGRO_FINAL = BASE_DIR / "data" / "processed" / "final_agro_graph.ttl"


In [46]:
# The namespace prefixes used in the SPARQL queries are defined as follows:
AGRO = Namespace("http://www.semanticweb.org/frank/ontologies/2025/7/agro_geokg#")
GEO = Namespace("http://www.opengis.net/ont/geosparql#")
XSD =  Namespace("http://www.w3.org/2001/XMLSchema#")

In [47]:
# Initialize the graph and parse the final Turtle file
g = Graph()
g.parse(str(INPUT_FILE_RDF_AGRO_FINAL), format="turtle")

<Graph identifier=N226abf78daef4d659f24f0efe4cdcd64 (<class 'rdflib.graph.Graph'>)>

In [48]:
# Example 1: Query all agricultural areas.
# The result is a table containing the URIs of all AgriculturalArea individuals.
# Example:
# Namespace URI = http://www.semanticweb.org/frank/ontologies/2025/7/agro_geokg#
# Individual URI = .../agro_geokg#agriculture_1
# Individual: an instance of the class AgriculturalArea
query = """

SELECT ?area
WHERE {
    ?area a agro:AgriculturalArea .
}

"""

results = g.query(query)

df = pd.DataFrame(results, columns=results.vars)
display(df.head(2))

,area
0,http://www.semanticweb.org/frank/ontologies/20...
1,http://www.semanticweb.org/frank/ontologies/20...


In [41]:
# Example 2: quering all agriculture areas with ndvi values
query = """

SELECT ?area ?ndvi
WHERE {
    ?area a agro:AgriculturalArea ;
          agro:meanNDVI ?ndvi .
}

"""

results = g.query(query)


df = pd.DataFrame(results, columns=[str(var) for var in results.vars])
display(df.head())

,area,ndvi
0,http://www.semanticweb.org/frank/ontologies/20...,0.549
1,http://www.semanticweb.org/frank/ontologies/20...,0.486
2,http://www.semanticweb.org/frank/ontologies/20...,0.384
3,http://www.semanticweb.org/frank/ontologies/20...,0.526
4,http://www.semanticweb.org/frank/ontologies/20...,0.524


In [43]:
# Replace the full namespace URI with the shorter AGRO prefix for better readability

df['area'] = df['area'].astype(str).str.replace(
    "http://www.semanticweb.org/frank/ontologies/2025/7/agro_geokg#",
    "agro:"
)

df.head()


,area,ndvi
0,agro:agriculture_1,0.549
1,agro:agriculture_10,0.486
2,agro:agriculture_100,0.384
3,agro:agriculture_101,0.526
4,agro:agriculture_102,0.524


The following SPARQL query demonstrates how information derived from a raster dataset and two different vector datasets can be queried together within a knowledge graph. It returns all agricultural areas located within the test area that have an NDVI value greater than 0.5 and an area larger than 20 hectares, together with all cycleways longer than 250 meters.The query result is returned as a table.

In [49]:
query = """

SELECT ?field ?area_in_ha ?ndvi ?cycleway ?length
WHERE {

  ?field a agro:AgriculturalArea ;
         agro:area_ha ?area_in_ha ;
         agro:meanNDVI ?ndvi ;
         geo:sfWithin agro:test_area_1 .

  ?cycleway a agro:OSMWay ;
            agro:highway "cycleway" ;
            agro:lengthMeters ?length ;
            geo:sfWithin agro:test_area_1 .

  FILTER(xsd:double(?area_in_ha) > 20)
  FILTER(xsd:double(?ndvi) > 0.5)
  FILTER(xsd:double(?length) > 250)

}
ORDER BY DESC(?ndvi)
"""

results = g.query(query)

for row in results:
    #print(row.area)
    continue

df = pd.DataFrame(results, columns=[str(var) for var in results.vars])

# Replace the full namespace URI with the shorter AGRO prefix for better readability

df['field'] = df['field'].astype(str).str.replace("http://www.semanticweb.org/frank/ontologies/2025/7/agro_geokg#", "agro: ")
df['cycleway'] = df['cycleway'].astype(str).str.replace("http://www.semanticweb.org/frank/ontologies/2025/7/agro_geokg#", "agro: ")

df.head(15)

,field,area_in_ha,ndvi,cycleway,length
0,agro: agriculture_207,24.24,0.555,agro: osm_way_1099,700.92
1,agro: agriculture_207,24.24,0.555,agro: osm_way_1100,252.9
2,agro: agriculture_207,24.24,0.555,agro: osm_way_1120,276.67
3,agro: agriculture_207,24.24,0.555,agro: osm_way_1597,604.26
4,agro: agriculture_207,24.24,0.555,agro: osm_way_753,360.81
5,agro: agriculture_207,24.24,0.555,agro: osm_way_987,1075.33
6,agro: agriculture_254,47.92,0.553,agro: osm_way_1099,700.92
7,agro: agriculture_254,47.92,0.553,agro: osm_way_1100,252.9
8,agro: agriculture_254,47.92,0.553,agro: osm_way_1120,276.67
9,agro: agriculture_254,47.92,0.553,agro: osm_way_1597,604.26
